# Batch 2 — Combined Training + Multi-Model Ensemble
### Training: `training_batch_with_labels.npz` + `first_batch_with_labels.npz` (2200 users)
### Test: `second_batch.npz` (860 users)

**Models:** XGBoost, CatBoost, LightGBM, Balanced Random Forest, SVM, Logistic Regression → Stacking Ensemble

In [ ]:
import pandas as pd
import numpy as np
import zipfile, warnings
import xgboost as xgb
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from imblearn.ensemble import BalancedRandomForestClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from feature_pipeline import (
    combine_labeled_data, load_data, compute_item_stats,
    build_features, get_feature_columns
)

print('Imports OK')

: 

## 1. Load & Combine Training Data + Build Features

In [ ]:
# Combine both labeled datasets for maximum training data
combined_path = combine_labeled_data(
    "data/training_batch_with_labels.npz",
    "data/first_batch_with_labels.npz"
)

# Load combined training + test
XX_train, yy, XX_test = load_data(combined_path, test_path="data/second_batch.npz")

print(f"\nTraining interactions: {XX_train.shape[0]:,}")
print(f"Training users: {yy.shape[0]} ({int(yy['label'].sum())} anomalous)")
print(f"Test interactions: {XX_test.shape[0]:,}")
print(f"Test users: {XX_test['user'].nunique()}")

In [ ]:
# Compute item stats from training ONLY (leakage-safe)
print("Computing item stats + SVD...")
item_stats = compute_item_stats(XX_train, n_svd_components=20)

# Build features for train and test
print("Building training features...")
train_feats = build_features(XX_train, item_stats).merge(yy, on="user")
print("Building test features...")
test_feats = build_features(XX_test, item_stats)

feature_cols = get_feature_columns(train_feats)

X_train = train_feats[feature_cols].values
y_train = train_feats["label"].values
X_test = test_feats[feature_cols].values
test_users = test_feats["user"].values

# Scale
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"\nFeatures: {len(feature_cols)}")
print(f"Training shape: {X_train_s.shape}")
print(f"Test shape: {X_test_s.shape}")
print(f"\nFeature list:")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:2d}. {f}")

## 2. Cross-Validation Setup + Helper Functions

In [ ]:
SEED = 42
N_FOLDS = 5
spw = np.sum(y_train == 0) / np.sum(y_train == 1)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Class imbalance ratio: {spw:.2f}")
print(f"Anomalous users: {np.sum(y_train == 1)}, Normal: {np.sum(y_train == 0)}")
print(f"Per fold: ~{np.sum(y_train == 1) // N_FOLDS} anomalous in validation")


def evaluate_oof(name, oof_scores, y_true, threshold=None):
    """Evaluate OOF predictions and find best F1 threshold."""
    auc = roc_auc_score(y_true, oof_scores)

    # Find best F1 threshold
    thresholds = np.linspace(0.01, 0.99, 500)
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (oof_scores >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    preds = (oof_scores >= best_t).astype(int)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  OOF AUC:     {auc:.4f}")
    print(f"  Best F1:     {best_f1:.4f} (threshold={best_t:.3f})")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    return auc, best_f1, best_t

## 3. Model 1 — XGBoost with Optuna

In [ ]:
def xgb_objective(trial):
    grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 3000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 3, 20),
        gamma             = trial.suggest_float('gamma', 0.0, 3.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        max_delta_step    = trial.suggest_int('max_delta_step', 0, 10),
        scale_pos_weight  = trial.suggest_float('scale_pos_weight', spw * 0.5, spw * 3.0),
        grow_policy       = grow_policy,
    )
    if grow_policy == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 128)
    else:
        params['max_depth'] = trial.suggest_int('max_depth', 3, 10)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    aucs = []
    for tr_i, val_i in cv.split(X_train_s, y_train):
        m = xgb.XGBClassifier(
            **params, eval_metric='aucpr', early_stopping_rounds=50,
            random_state=SEED, n_jobs=-1, tree_method='hist'
        )
        m.fit(X_train_s[tr_i], y_train[tr_i],
              eval_set=[(X_train_s[val_i], y_train[val_i])], verbose=False)
        aucs.append(roc_auc_score(y_train[val_i],
                                   m.predict_proba(X_train_s[val_i])[:, 1]))
    return np.mean(aucs)

print("Running XGBoost Optuna search (200 trials)...")
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=200, show_progress_bar=True)

xgb_best = xgb_study.best_params
print(f'\nBest XGBoost CV AUC: {xgb_study.best_value:.4f}')
print('Best params:', xgb_best)

In [ ]:
# XGBoost OOF predictions
oof_xgb = np.zeros(len(y_train))
test_preds_xgb = np.zeros(len(X_test_s))
xgb_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = xgb.XGBClassifier(
        **xgb_best, eval_metric='aucpr', early_stopping_rounds=50,
        random_state=SEED, n_jobs=-1, tree_method='hist'
    )
    m.fit(X_train_s[tr_idx], y_train[tr_idx],
          eval_set=[(X_train_s[val_idx], y_train[val_idx])], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_xgb += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    xgb_models.append(m)
    print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_xgb[val_idx]):.4f}")

xgb_auc, xgb_f1, xgb_thresh = evaluate_oof("XGBoost", oof_xgb, y_train)

## 4. Model 2 — CatBoost with Optuna

In [ ]:
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not installed. Run: pip install catboost")

if HAS_CATBOOST:
    def cat_objective(trial):
        params = dict(
            iterations     = trial.suggest_int('iterations', 500, 3000),
            learning_rate  = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            depth          = trial.suggest_int('depth', 4, 8),
            l2_leaf_reg    = trial.suggest_float('l2_leaf_reg', 0.5, 10.0),
            bagging_temperature = trial.suggest_float('bagging_temperature', 0, 3.0),
            random_strength     = trial.suggest_float('random_strength', 0.5, 5.0),
            border_count        = trial.suggest_int('border_count', 32, 255),
            auto_class_weights  = 'Balanced',
            eval_metric         = 'AUC',
            random_seed         = SEED,
            verbose             = 0,
        )
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
        aucs = []
        for tr_i, val_i in cv.split(X_train_s, y_train):
            m = CatBoostClassifier(**params)
            m.fit(X_train_s[tr_i], y_train[tr_i],
                  eval_set=(X_train_s[val_i], y_train[val_i]),
                  early_stopping_rounds=50, verbose=0)
            aucs.append(roc_auc_score(y_train[val_i],
                                       m.predict_proba(X_train_s[val_i])[:, 1]))
        return np.mean(aucs)

    print("Running CatBoost Optuna search (100 trials)...")
    cat_study = optuna.create_study(direction='maximize')
    cat_study.optimize(cat_objective, n_trials=100, show_progress_bar=True)
    cat_best = cat_study.best_params
    print(f'\nBest CatBoost CV AUC: {cat_study.best_value:.4f}')
    print('Best params:', cat_best)

In [ ]:
# CatBoost OOF predictions
if HAS_CATBOOST:
    oof_cat = np.zeros(len(y_train))
    test_preds_cat = np.zeros(len(X_test_s))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
        m = CatBoostClassifier(
            **cat_best,
            auto_class_weights='Balanced',
            eval_metric='AUC',
            random_seed=SEED,
            verbose=0,
        )
        m.fit(X_train_s[tr_idx], y_train[tr_idx],
              eval_set=(X_train_s[val_idx], y_train[val_idx]),
              early_stopping_rounds=50, verbose=0)
        oof_cat[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
        test_preds_cat += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
        print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_cat[val_idx]):.4f}")

    cat_auc, cat_f1, cat_thresh = evaluate_oof("CatBoost", oof_cat, y_train)
else:
    oof_cat = oof_xgb.copy()  # fallback
    test_preds_cat = test_preds_xgb.copy()
    print("Skipping CatBoost (not installed)")

## 5. Model 3 — LightGBM with Optuna

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 3000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 16, 128),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        scale_pos_weight  = trial.suggest_float('scale_pos_weight', spw * 0.5, spw * 3.0),
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    aucs = []
    for tr_i, val_i in cv.split(X_train_s, y_train):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_train_s[tr_i], y_train[tr_i],
              eval_set=[(X_train_s[val_i], y_train[val_i])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        aucs.append(roc_auc_score(y_train[val_i],
                                   m.predict_proba(X_train_s[val_i])[:, 1]))
    return np.mean(aucs)

print("Running LightGBM Optuna search (150 trials)...")
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=150, show_progress_bar=True)

lgb_best = lgb_study.best_params
print(f'\nBest LightGBM CV AUC: {lgb_study.best_value:.4f}')
print('Best params:', lgb_best)

In [ ]:
# LightGBM OOF predictions
oof_lgb = np.zeros(len(y_train))
test_preds_lgb = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = lgb.LGBMClassifier(**lgb_best, random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(X_train_s[tr_idx], y_train[tr_idx],
          eval_set=[(X_train_s[val_idx], y_train[val_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_lgb += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_lgb[val_idx]):.4f}")

lgb_auc, lgb_f1, lgb_thresh = evaluate_oof("LightGBM", oof_lgb, y_train)

## 6. Model 4 — Balanced Random Forest

In [ ]:
# Balanced Random Forest — undersamples majority per bootstrap
oof_brf = np.zeros(len(y_train))
test_preds_brf = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = BalancedRandomForestClassifier(
        n_estimators=500, max_depth=10, min_samples_leaf=5,
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_train_s[tr_idx], y_train[tr_idx])
    oof_brf[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_brf += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_brf[val_idx]):.4f}")

brf_auc, brf_f1, brf_thresh = evaluate_oof("Balanced RF", oof_brf, y_train)

## 7. Model 5 — SVM (RBF Kernel)

In [ ]:
# SVM with RBF kernel — use StandardScaler for SVM
scaler_svm = StandardScaler()
X_train_svm = scaler_svm.fit_transform(X_train)
X_test_svm = scaler_svm.transform(X_test)

oof_svm = np.zeros(len(y_train))
test_preds_svm = np.zeros(len(X_test_svm))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_svm, y_train)):
    m = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced',
            probability=True, random_state=SEED)
    m.fit(X_train_svm[tr_idx], y_train[tr_idx])
    oof_svm[val_idx] = m.predict_proba(X_train_svm[val_idx])[:, 1]
    test_preds_svm += m.predict_proba(X_test_svm)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_svm[val_idx]):.4f}")

svm_auc, svm_f1, svm_thresh = evaluate_oof("SVM (RBF)", oof_svm, y_train)

## 8. Model 6 — Logistic Regression (baseline + meta-learner candidate)

In [ ]:
# Logistic Regression
oof_lr = np.zeros(len(y_train))
test_preds_lr = np.zeros(len(X_test_svm))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_svm, y_train)):
    m = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=SEED)
    m.fit(X_train_svm[tr_idx], y_train[tr_idx])
    oof_lr[val_idx] = m.predict_proba(X_train_svm[val_idx])[:, 1]
    test_preds_lr += m.predict_proba(X_test_svm)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_lr[val_idx]):.4f}")

lr_auc, lr_f1, lr_thresh = evaluate_oof("Logistic Regression", oof_lr, y_train)

## 9. Model Comparison Summary

In [ ]:
# Summary of all models
results = pd.DataFrame({
    'Model': ['XGBoost', 'CatBoost' if HAS_CATBOOST else 'CatBoost (skipped)',
              'LightGBM', 'Balanced RF', 'SVM (RBF)', 'Logistic Regression'],
    'OOF AUC': [roc_auc_score(y_train, oof_xgb),
                roc_auc_score(y_train, oof_cat),
                roc_auc_score(y_train, oof_lgb),
                roc_auc_score(y_train, oof_brf),
                roc_auc_score(y_train, oof_svm),
                roc_auc_score(y_train, oof_lr)],
})
results = results.sort_values('OOF AUC', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 10. Stacking Ensemble (Meta-Learner)

In [ ]:
# Stack OOF predictions as meta-features
oof_stack = np.column_stack([oof_xgb, oof_cat, oof_lgb, oof_brf, oof_svm, oof_lr])
test_stack = np.column_stack([test_preds_xgb, test_preds_cat, test_preds_lgb,
                               test_preds_brf, test_preds_svm, test_preds_lr])

print(f"Stacking meta-features shape: {oof_stack.shape}")
print(f"Test meta-features shape: {test_stack.shape}")

# Train meta-learner (Logistic Regression) on OOF predictions
oof_meta = np.zeros(len(y_train))
test_meta = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_stack, y_train)):
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    meta.fit(oof_stack[tr_idx], y_train[tr_idx])
    oof_meta[val_idx] = meta.predict_proba(oof_stack[val_idx])[:, 1]
    test_meta += meta.predict_proba(test_stack)[:, 1] / N_FOLDS
    print(f"  Meta Fold {fold+1}: AUC={roc_auc_score(y_train[val_idx], oof_meta[val_idx]):.4f}")

meta_auc, meta_f1, meta_thresh = evaluate_oof("Stacking Ensemble", oof_meta, y_train)

In [ ]:
# Also try simple rank-average blend (no meta-learner)
from scipy.stats import rankdata

def rank_blend(*pred_arrays):
    """Rank-average blend of multiple prediction arrays."""
    ranks = [rankdata(p) for p in pred_arrays]
    return np.mean(ranks, axis=0)

# Rank-average of top 3 boosting models
blend_boost = rank_blend(test_preds_xgb, test_preds_cat, test_preds_lgb)
oof_blend_boost = rank_blend(oof_xgb, oof_cat, oof_lgb)
blend_boost_auc = roc_auc_score(y_train, oof_blend_boost)

# Rank-average of all 6 models
blend_all = rank_blend(test_preds_xgb, test_preds_cat, test_preds_lgb,
                       test_preds_brf, test_preds_svm, test_preds_lr)
oof_blend_all = rank_blend(oof_xgb, oof_cat, oof_lgb, oof_brf, oof_svm, oof_lr)
blend_all_auc = roc_auc_score(y_train, oof_blend_all)

print(f"\nRank-average blend (3 boosters): OOF AUC = {blend_boost_auc:.4f}")
print(f"Rank-average blend (all 6):      OOF AUC = {blend_all_auc:.4f}")
print(f"Stacking meta-learner:           OOF AUC = {meta_auc:.4f}")

# Pick the best approach
best_approaches = {
    'Stacking Meta': (test_meta, meta_auc),
    'Rank Blend (3 boosters)': (blend_boost, blend_boost_auc),
    'Rank Blend (all 6)': (blend_all, blend_all_auc),
    'XGBoost only': (test_preds_xgb, roc_auc_score(y_train, oof_xgb)),
}
best_name = max(best_approaches, key=lambda k: best_approaches[k][1])
best_test_preds = best_approaches[best_name][0]
best_auc = best_approaches[best_name][1]
print(f"\n>>> Best approach: {best_name} (OOF AUC = {best_auc:.4f})")

## 11. Feature Importance (XGBoost)

In [ ]:
# Average feature importance across XGBoost fold models
avg_imp = np.mean([m.feature_importances_ for m in xgb_models], axis=0)
feat_imp = pd.Series(avg_imp, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
feat_imp.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title('XGBoost Average Feature Importance (5-fold)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 15 features:")
print(feat_imp.head(15).to_string())

## 12. Generate Submission for second_batch

In [ ]:
# Sort predictions by user ID (Codabench expects this order)
test_order = test_feats.sort_values('user').reset_index(drop=True)
sort_idx = test_feats['user'].argsort().values

# Use best approach's predictions, reordered by user ID
y_score = best_test_preds[sort_idx]

# Normalise to [0, 1]
y_score_norm = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)

# Save submission
np.savez('submission.npz', predictions=y_score_norm)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('submission.npz', arcname='submission.npz')

print(f"Submission method: {best_name}")
print(f"Users predicted:   {len(y_score_norm)}")
print(f"First 5 user IDs:  {test_order['user'].values[:5]}")
print(f"Score range:       [{y_score_norm.min():.4f}, {y_score_norm.max():.4f}]")
print(f"Score mean:        {y_score_norm.mean():.4f}")
print(f"\nsubmission.zip ready for Codabench")

In [ ]:
# Save alternative submissions (in case best approach overfits on Codabench)
import os
os.makedirs('submission_files', exist_ok=True)

alternatives = {
    'xgboost': test_preds_xgb,
    'catboost': test_preds_cat,
    'lightgbm': test_preds_lgb,
    'stacking': test_meta,
    'rank_blend_3boost': blend_boost,
    'rank_blend_all6': blend_all,
}

for name, preds in alternatives.items():
    p = preds[sort_idx]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    np.savez(f'submission_files/{name}.npz', predictions=p_norm)
    with zipfile.ZipFile(f'submission_files/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(f'submission_files/{name}.npz', arcname='submission.npz')

print("Alternative submissions saved in submission_files/:")
for name in alternatives:
    print(f"  {name}.zip")